# MBEC V11B — Targeted online-safe stream rerun

This notebook refits only four manuscript configurations:

- RF-F1;
- RF-F7;
- XGB-F1;
- XGB-F6.

It saves every stream prediction and compares:

1. the original V10 label-gated alarm generator; and
2. an **online-safe gap-only** generator that does not inspect Phase,
   TargetPreictal, or EligibleForFAR while producing alarms.

Retrospective labels are used only after alarm generation to score seizure
sensitivity, FAR/h, and TTW. Internal dataset and column names containing
`wPLI` are preserved for compatibility; manuscript-facing output uses
**pd-wPLI**.


## 0–4. V10-compatible loading, preprocessing, fitting, and alarm primitives


In [ ]:
# ==========================================================================
# 0. IMPORTS AND LOCKED V10 CONFIGURATION
# ==========================================================================
import gc
import json
import math
import re
import shutil
import time
import traceback
from collections import deque
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
from IPython.display import display
from scipy.stats import friedmanchisquare, wilcoxon
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import confusion_matrix, make_scorer, matthews_corrcoef
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

try:
    import xgboost
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError('V10 requires xgboost, which is normally preinstalled on Kaggle') from exc

RANDOM_SEED = 42
EPOCH_SEC = 10
SPH_SEC = 0
SOP_SEC = 30 * 60
REFRACTORY_SEC = 30 * 60
PRIMARY_ALARM_NAME = 'FP_5min_theta_0.70'
PRIMARY_FP_WINDOW_EPOCHS = 30
PRIMARY_FP_THRESHOLD = 0.70
N_SELECTED_FEATURES = 20
TRAIN_NEG_POS_RATIO = 1.0

MODEL_ORDER = ['RandomForest', 'XGBoost']
INNER_CV_MAX_SPLITS = 3
INNER_SCORING = 'MCC'
GRID_N_JOBS = 2

# F1-F4 retain V8's global Top-20 rule. Quotas apply only to mixed blocks.
BLOCK_BALANCED_QUOTAS = {
    ('Handcrafted', 'Graph_PLI'): {'Handcrafted': 10, 'Graph_PLI': 10},
    ('Handcrafted', 'Graph_wPLI'): {'Handcrafted': 10, 'Graph_wPLI': 10},
    ('Handcrafted', 'Graph_PLI', 'Graph_wPLI'): {
        'Handcrafted': 10, 'Graph_PLI': 5, 'Graph_wPLI': 5,
    },
}

ALARM_SENSITIVITY_SETTINGS = {
    'FP_2min_theta_0.70': {'mode': 'firing_power', 'window_epochs': 12, 'threshold': 0.70},
    PRIMARY_ALARM_NAME: {'mode': 'firing_power', 'window_epochs': 30, 'threshold': 0.70},
    'FP_10min_theta_0.70': {'mode': 'firing_power', 'window_epochs': 60, 'threshold': 0.70},
    'FP_5min_theta_0.50': {'mode': 'firing_power', 'window_epochs': 30, 'threshold': 0.50},
    'FP_5min_theta_0.80': {'mode': 'firing_power', 'window_epochs': 30, 'threshold': 0.80},
    'Legacy_90_consecutive': {'mode': 'consecutive', 'k': 90},
}
assert ALARM_SENSITIVITY_SETTINGS[PRIMARY_ALARM_NAME] == {
    'mode': 'firing_power',
    'window_epochs': PRIMARY_FP_WINDOW_EPOCHS,
    'threshold': PRIMARY_FP_THRESHOLD,
}

INPUT_SEARCH_ROOT = Path('/kaggle/input')
MODEL_READY_ROOT_OVERRIDE = None
OUTPUT_ROOT = Path('/kaggle/working/MBEC_V11B_TARGETED_STREAM')

# First run: True. To resume after interruption without deleting completed
# subjects, change to False and rerun the notebook from the beginning.
RESET_OUTPUT = False

RF_PARAMS = {
    'n_estimators': 500,
    'max_depth': 15,
    'min_samples_leaf': 4,
    'max_features': 'sqrt',
    'class_weight': None,
    'random_state': RANDOM_SEED,
    'n_jobs': -1,
}

XGB_BASE_PARAMS = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 1,
    'reg_lambda': 1.0,
    'random_state': RANDOM_SEED,
    'tree_method': 'hist',
    'n_jobs': 1,
}

# List-of-dictionaries prevents an unintended Cartesian product and keeps the
# nested computation feasible. Each dictionary is one pre-declared candidate.
XGB_PARAM_GRID = [
    {'n_estimators': [200], 'max_depth': [3], 'learning_rate': [0.05]},
    {'n_estimators': [400], 'max_depth': [3], 'learning_rate': [0.03]},
    {'n_estimators': [300], 'max_depth': [5], 'learning_rate': [0.05]},
    {'n_estimators': [200], 'max_depth': [5], 'learning_rate': [0.10]},
]

SVM_BASE_PARAMS = {
    'kernel': 'rbf',
    'class_weight': None,
    'probability': False,
    'cache_size': 1024,
}
SVM_PARAM_GRID = [
    {'C': [0.1], 'gamma': ['scale']},
    {'C': [1.0], 'gamma': ['scale']},
    {'C': [10.0], 'gamma': ['scale']},
    {'C': [1.0], 'gamma': [0.1]},
    {'C': [10.0], 'gamma': [0.01]},
]

BLOCK_PREFIX = {
    'Handcrafted': 'HC_',
    'Graph_PLI': 'PLI_',
    'Graph_wPLI': 'wPLI_',
}



# ==========================================================================
# 1. DISCOVER AND AUDIT THE MODEL-READY DATASET
# ==========================================================================
def discover_model_ready_root():
    if MODEL_READY_ROOT_OVERRIDE is not None:
        candidates = [Path(MODEL_READY_ROOT_OVERRIDE)]
    else:
        candidates = list(INPUT_SEARCH_ROOT.rglob('ModelReady_GAPAWARE_V4'))
    valid = []
    for root in candidates:
        required = [root / 'Metadata', root / 'Handcrafted', root / 'Graph_PLI',
                    root / 'Graph_wPLI', root / 'feature_set_definitions.json']
        if all(p.exists() for p in required):
            valid.append(root.resolve())
    valid = sorted(set(valid))
    if not valid:
        raise FileNotFoundError('Could not find a complete ModelReady_GAPAWARE_V4 dataset')
    if len(valid) > 1:
        raise RuntimeError(f'Multiple model-ready roots found: {valid}; set MODEL_READY_ROOT_OVERRIDE')
    return valid[0]


MODEL_ROOT = discover_model_ready_root()
with open(MODEL_ROOT / 'feature_set_definitions.json', encoding='utf-8') as file:
    FEATURE_SET_DEFINITIONS = json.load(file)

subjects = sorted(p.name for p in (MODEL_ROOT / 'Metadata').iterdir()
                  if p.is_dir() and re.fullmatch(r'chb\d{2}', p.name))
if not subjects:
    raise RuntimeError('No model-ready subjects found')

print('Model-ready root:', MODEL_ROOT)
print(f'Subjects ({len(subjects)}):', subjects)
print('Feature sets:', list(FEATURE_SET_DEFINITIONS))




# ==========================================================================
# 2. LOAD ONE SUBJECT AND ALIGN ALL THREE FEATURE BLOCKS
# ==========================================================================
def is_testing_stem(stem):
    return re.search(r'_block_\d+$', stem) is not None


def keys_for_metadata(meta):
    if 'Class' in meta.columns:
        return ['Patient', 'Class', 'SeizureID', 'Epoch']
    return ['Patient', 'SeizureID', 'Block', 'Epoch']


def feature_columns_for_blocks(frame, blocks):
    prefixes = tuple(BLOCK_PREFIX[b] for b in blocks)
    cols = [c for c in frame.columns if c.startswith(prefixes)]
    if not cols:
        raise ValueError(f'No features found for blocks={blocks}')
    return cols


def load_subject(subject):
    meta_files = sorted((MODEL_ROOT / 'Metadata' / subject).glob('*.parquet'))
    if not meta_files:
        raise FileNotFoundError(f'No metadata files for {subject}')
    training, streams = [], []

    for meta_path in meta_files:
        stem = meta_path.stem
        meta = pd.read_parquet(meta_path)
        keys = keys_for_metadata(meta)
        if meta.duplicated(keys).any():
            raise AssertionError(f'{subject}/{stem}: duplicate metadata epoch keys')
        frame = meta.copy()
        for block in BLOCK_PREFIX:
            path = MODEL_ROOT / block / subject / f'{stem}.parquet'
            if not path.exists():
                raise FileNotFoundError(path)
            features = pd.read_parquet(path)
            if features.duplicated(keys).any():
                raise AssertionError(f'{block}/{subject}/{stem}: duplicate keys')
            feature_cols = [c for c in features.columns if c not in keys]
            frame = frame.merge(features[keys + feature_cols], on=keys,
                                how='inner', validate='one_to_one')
        if len(frame) != len(meta):
            raise AssertionError(f'{subject}/{stem}: feature merge lost epochs')
        (streams if is_testing_stem(stem) else training).append(frame)

    train_pool = pd.concat(training, ignore_index=True)
    stream_pool = pd.concat(streams, ignore_index=True)
    train_pool['Class'] = train_pool['Class'].map({'interictal': 0, 'preictal': 1})
    if train_pool['Class'].isna().any():
        raise ValueError(f'{subject}: unknown training class')
    train_pool['Class'] = train_pool['Class'].astype(int)
    stream_pool['EpochStart'] = pd.to_datetime(stream_pool['EpochStart'], utc=True)

    # Strong cohort invariants inherited from segmentation.
    for sid, group in train_pool.groupby('SeizureID'):
        counts = group['Class'].value_counts().to_dict()
        if counts.get(1, 0) != 180 or counts.get(0, 0) != 540:
            raise AssertionError(f'{subject}/seizure {sid}: expected 180/540, got {counts}')
        target = stream_pool.loc[stream_pool['SeizureID'] == sid]
        if int(target['TargetPreictal'].sum()) != 180:
            raise AssertionError(f'{subject}/seizure {sid}: continuous preictal != 180 epochs')
        if target.duplicated(['Epoch']).any():
            raise AssertionError(f'{subject}/seizure {sid}: duplicate continuous Epoch')
    return train_pool, stream_pool




# ==========================================================================
# 3. TRAINING-ONLY BALANCING, SCALING, MI SELECTION AND INNER TUNING
# ==========================================================================
def balance_training(frame, fold_seed):
    positive = frame.loc[frame['Class'] == 1]
    negative = frame.loc[frame['Class'] == 0]
    if positive.empty or negative.empty:
        raise ValueError('Training fold must contain both classes')
    n_negative = min(len(negative), int(round(len(positive) * TRAIN_NEG_POS_RATIO)))
    negative = negative.sample(n=n_negative, replace=False, random_state=fold_seed)
    return (pd.concat([positive, negative], ignore_index=True)
              .sample(frac=1.0, random_state=fold_seed)
              .reset_index(drop=True))


def feature_block(feature_name):
    matches = [block for block, prefix in BLOCK_PREFIX.items()
               if feature_name.startswith(prefix)]
    if len(matches) != 1:
        raise ValueError(f'Cannot assign feature to exactly one block: {feature_name}')
    return matches[0]


def feature_selection_plan(blocks):
    block_set = set(blocks)
    for configured_blocks, quotas in BLOCK_BALANCED_QUOTAS.items():
        if len(blocks) == len(configured_blocks) and block_set == set(configured_blocks):
            if sum(quotas.values()) != N_SELECTED_FEATURES:
                raise AssertionError(f'Block quotas do not sum to {N_SELECTED_FEATURES}')
            return 'block_balanced_mi', dict(quotas)
    return 'global_top_mi', {'GLOBAL': N_SELECTED_FEATURES}


def select_feature_indices(feature_cols, mi_scores, blocks):
    mode, plan = feature_selection_plan(blocks)
    scores = np.asarray(mi_scores, dtype=float)
    if len(scores) != len(feature_cols):
        raise ValueError('MI-score/feature length mismatch')

    if mode == 'global_top_mi':
        k = min(N_SELECTED_FEATURES, len(feature_cols))
        selected_idx = np.sort(np.argsort(scores, kind='mergesort')[-k:])
        ranked = sorted(selected_idx, key=lambda i: (-scores[i], feature_cols[i]))
        records = []
        for rank, idx in enumerate(ranked, start=1):
            records.append({
                'Feature': feature_cols[idx], 'MIScore': float(scores[idx]),
                'FeatureBlock': feature_block(feature_cols[idx]),
                'RankWithinBlock': np.nan, 'RequestedBlockQuota': np.nan,
                'RankWithinSelected': rank,
            })
        return np.asarray(selected_idx, dtype=np.int64), records, mode, plan

    selected_idx, records = [], []
    for block in blocks:
        quota = int(plan[block])
        candidates = [i for i, name in enumerate(feature_cols)
                      if feature_block(name) == block]
        if len(candidates) < quota:
            raise ValueError(f'{block}: requested {quota} features, found {len(candidates)}')
        local = np.argsort(scores[candidates], kind='mergesort')[-quota:]
        chosen = [candidates[i] for i in local]
        ranked = sorted(chosen, key=lambda i: (-scores[i], feature_cols[i]))
        for block_rank, idx in enumerate(ranked, start=1):
            selected_idx.append(idx)
            records.append({
                'Feature': feature_cols[idx], 'MIScore': float(scores[idx]),
                'FeatureBlock': block, 'RankWithinBlock': block_rank,
                'RequestedBlockQuota': quota,
                'RankWithinSelected': len(records) + 1,
            })

    if len(selected_idx) != N_SELECTED_FEATURES or len(set(selected_idx)) != len(selected_idx):
        raise AssertionError('Block-balanced selector did not return 20 unique features')
    return (np.sort(np.asarray(selected_idx, dtype=np.int64)), records,
            mode, plan)


def prepare_fold(train, test, stream, feature_cols, blocks, fold_seed):
    balanced = balance_training(train, fold_seed)
    X_train = balanced[feature_cols].to_numpy(dtype=np.float32)
    y_train = balanced['Class'].to_numpy(dtype=np.int8)
    groups_train = balanced['SeizureID'].to_numpy(dtype=np.int32)
    X_test = test[feature_cols].to_numpy(dtype=np.float32)
    y_test = test['Class'].to_numpy(dtype=np.int8)
    X_stream = stream[feature_cols].to_numpy(dtype=np.float32)

    for name, array in [('train', X_train), ('test', X_test), ('stream', X_stream)]:
        if not np.isfinite(array).all():
            raise ValueError(f'{name}: NaN or Inf before modelling')

    scaler = StandardScaler()
    X_train_z = scaler.fit_transform(X_train)
    X_test_z = scaler.transform(X_test)
    X_stream_z = scaler.transform(X_stream)

    mi_scores = mutual_info_classif(X_train_z, y_train, random_state=RANDOM_SEED)
    selected_idx, selected, selection_mode, selection_plan = select_feature_indices(
        feature_cols, mi_scores, blocks)
    return {
        'X_train': X_train_z[:, selected_idx], 'y_train': y_train,
        'groups_train': groups_train,
        'X_test': X_test_z[:, selected_idx], 'y_test': y_test,
        'X_stream': X_stream_z[:, selected_idx],
        'selected': selected, 'balanced_n': len(balanced),
        'scaler': scaler, 'selected_idx': selected_idx,
        'selection_mode': selection_mode, 'selection_plan': selection_plan,
    }


def _jsonable_params(params):
    clean = {}
    for key, value in params.items():
        clean[key] = value.item() if isinstance(value, np.generic) else value
    return clean


def fit_classifier(model_name, X, y, groups, fold_seed):
    start = time.perf_counter()
    if model_name == 'RandomForest':
        model = RandomForestClassifier(**RF_PARAMS)
        model.fit(X, y)
        summary = {
            'TuningMode': 'locked_V9A_parameters', 'InnerSplits': 0,
            'BestInnerMCC': np.nan, 'BestInnerMCC_SD': np.nan,
            'BestParams': json.dumps(_jsonable_params(RF_PARAMS), sort_keys=True),
        }
        candidates = []
    else:
        unique_groups = np.unique(groups)
        n_splits = min(INNER_CV_MAX_SPLITS, len(unique_groups))
        if n_splits < 2:
            raise ValueError('Grouped inner CV requires at least two training seizures')
        cv = StratifiedGroupKFold(
            n_splits=n_splits, shuffle=True, random_state=fold_seed)
        scorer = make_scorer(matthews_corrcoef)
        if model_name == 'XGBoost':
            estimator = XGBClassifier(**XGB_BASE_PARAMS)
            grid = XGB_PARAM_GRID
        elif model_name == 'SVM_RBF':
            estimator = SVC(**SVM_BASE_PARAMS)
            grid = SVM_PARAM_GRID
        else:
            raise ValueError(f'Unknown model: {model_name}')
        search = GridSearchCV(
            estimator=estimator, param_grid=grid, scoring=scorer, cv=cv,
            refit=True, n_jobs=GRID_N_JOBS, error_score='raise',
            return_train_score=False)
        search.fit(X, y, groups=groups)
        model = search.best_estimator_
        best_index = int(search.best_index_)
        summary = {
            'TuningMode': 'grouped_inner_CV', 'InnerSplits': n_splits,
            'BestInnerMCC': float(search.cv_results_['mean_test_score'][best_index]),
            'BestInnerMCC_SD': float(search.cv_results_['std_test_score'][best_index]),
            'BestParams': json.dumps(_jsonable_params(search.best_params_), sort_keys=True),
        }
        candidates = []
        for idx, params in enumerate(search.cv_results_['params']):
            candidates.append({
                'CandidateIndex': idx,
                'Params': json.dumps(_jsonable_params(params), sort_keys=True),
                'MeanInnerMCC': float(search.cv_results_['mean_test_score'][idx]),
                'SDInnerMCC': float(search.cv_results_['std_test_score'][idx]),
                'RankInnerMCC': int(search.cv_results_['rank_test_score'][idx]),
                'Selected': int(idx == best_index),
            })
    summary['FitAndTuneTime_sec'] = time.perf_counter() - start
    return model, summary, candidates


def epoch_fingerprints(frame, fingerprint_cols):
    return pd.util.hash_pandas_object(
        frame[fingerprint_cols], index=False, categorize=False
    ).to_numpy(dtype=np.uint64)


def build_interictal_owner_map(train_pool, fingerprint_cols):
    negative = train_pool.loc[train_pool['Class'] == 0]
    hashes = epoch_fingerprints(negative, fingerprint_cols)
    owners = negative['SeizureID'].to_numpy(dtype=int)
    mapping = pd.DataFrame({'Hash': hashes, 'Owner': owners})
    conflicts = mapping.groupby('Hash')['Owner'].nunique()
    if (conflicts > 1).any():
        raise AssertionError('An interictal fingerprint belongs to multiple LOSO folds')
    return mapping.drop_duplicates('Hash').set_index('Hash')['Owner'].to_dict()


def predict_pipeline(frame, feature_cols, fitted):
    scaler, selected_idx, model = fitted
    values = frame[feature_cols].to_numpy(dtype=np.float32)
    if not np.isfinite(values).all():
        raise ValueError('NaN or Inf during cross-fitted stream prediction')
    transformed = scaler.transform(values)[:, selected_idx]
    return model.predict(transformed).astype(np.int8)



# ==========================================================================
# 4. CAUSAL CONTINUOUS ALARM, FAR/H AND TIME-TO-WARNING
# ==========================================================================
def _validate_alarm_setting(setting):
    mode = setting.get('mode')
    if mode == 'firing_power':
        window = int(setting['window_epochs'])
        threshold = float(setting['threshold'])
        if window < 1 or not (0 < threshold <= 1):
            raise ValueError(f'Invalid firing-power setting: {setting}')
        return mode, window, threshold, math.ceil(window * threshold)
    if mode == 'consecutive':
        k = int(setting['k'])
        if k < 1:
            raise ValueError(f'Invalid consecutive setting: {setting}')
        return mode, k, 1.0, k
    raise ValueError(f'Unknown alarm mode: {mode}')


def run_causal_alarm(y_pred, stream, setting):
    stream = stream.reset_index(drop=True)
    predictions = np.asarray(y_pred, dtype=np.int8)
    if len(predictions) != len(stream):
        raise ValueError('Prediction/stream length mismatch')
    if not np.isin(predictions, [0, 1]).all():
        raise ValueError('Alarm input predictions must be binary')

    mode, window_epochs, threshold, required_positive = _validate_alarm_setting(setting)

    history = deque(maxlen=window_epochs)
    refractory_until = None
    previous_start = None
    alarms = []

    for idx, (pred, row) in enumerate(zip(predictions, stream.itertuples(index=False))):
        start = row.EpochStart
        end = start + pd.Timedelta(seconds=EPOCH_SEC)
        gap = bool(row.Discontinuity)
        if previous_start is not None:
            elapsed = (start - previous_start).total_seconds()
            gap = gap or elapsed > EPOCH_SEC + 0.5 or elapsed <= 0
        previous_start = start

        phase = str(row.Phase).lower()
        valid_state = phase in {'interictal', 'preictal'}
        if gap or not valid_state:
            history.clear()
            continue
        if refractory_until is not None and end < refractory_until:
            history.clear()
            continue

        if mode == 'consecutive':
            if pred == 0:
                history.clear()
            else:
                history.append(1)
        else:
            history.append(int(pred))

        full_window = len(history) == window_epochs
        positive_count = int(sum(history))
        if full_window and positive_count >= required_positive:
            alarms.append({
                'Index': idx,
                'AlarmTime': end,
                'Phase': phase,
                'EligibleForFAR': int(row.EligibleForFAR),
                'TargetPreictal': int(row.TargetPreictal),
                'WindowEpochs': window_epochs,
                'PositiveEpochs': positive_count,
                'FiringPower': positive_count / window_epochs,
            })
            history.clear()
            refractory_until = end + pd.Timedelta(seconds=REFRACTORY_SEC)
    return alarms


def evaluate_alarm(y_pred, stream, setting):
    stream = stream.sort_values('Epoch', kind='stable').reset_index(drop=True)
    target = stream.loc[stream['TargetPreictal'].astype(bool)]
    if len(target) != 180:
        raise AssertionError(f'Expected 180 target-preictal epochs, got {len(target)}')
    seizure_onset = target['EpochStart'].max() + pd.Timedelta(seconds=EPOCH_SEC)
    valid_start = seizure_onset - pd.Timedelta(seconds=SPH_SEC + SOP_SEC)
    valid_end = seizure_onset - pd.Timedelta(seconds=SPH_SEC)

    alarms = run_causal_alarm(y_pred, stream, setting)
    true_alarms = [a for a in alarms if valid_start <= a['AlarmTime'] < valid_end]
    late_alarms = [a for a in alarms if valid_end <= a['AlarmTime'] < seizure_onset]
    false_alarms = [a for a in alarms if a['EligibleForFAR'] == 1]
    eligible_hours = float(stream['EligibleForFAR'].astype(bool).sum() * EPOCH_SEC / 3600.0)
    far_h = len(false_alarms) / eligible_hours if eligible_hours > 0 else np.nan
    ttw = ((seizure_onset - true_alarms[0]['AlarmTime']).total_seconds() / 60.0
           if true_alarms else np.nan)
    return {
        'SeizurePredicted': int(bool(true_alarms)),
        'FalseAlarms': len(false_alarms),
        'LateAlarms': len(late_alarms),
        'EligibleInterictalHours': eligible_hours,
        'FAR_per_h': far_h,
        'TimeToWarning_min': ttw,
        'TotalAlarms': len(alarms),
    }


def run_alarm_unit_tests():
    onset = pd.Timestamp('2076-01-01T01:00:00Z')
    starts = pd.date_range(onset - pd.Timedelta(minutes=40), periods=240, freq='10s')
    stream = pd.DataFrame({
        'Epoch': np.arange(240), 'EpochStart': starts,
        'Phase': ['interictal'] * 60 + ['preictal'] * 180,
        'TargetPreictal': [0] * 60 + [1] * 180,
        'EligibleForFAR': [1] * 60 + [0] * 180,
        'Discontinuity': 0,
    })
    predictions = np.array([0] * 60 + [1] * 180)
    primary = ALARM_SENSITIVITY_SETTINGS[PRIMARY_ALARM_NAME]
    result = evaluate_alarm(predictions, stream, primary)
    assert result['SeizurePredicted'] == 1
    # The 5-min causal window already contains nine preceding interictal
    # negatives, so the threshold is crossed after 21 positive epochs (3.5 min).
    assert abs(result['TimeToWarning_min'] - 26.5) < 1e-9
    assert result['FalseAlarms'] == 0

    # Nine isolated negatives are tolerated: 21/30 still reaches theta=0.70.
    noisy = predictions.copy()
    noisy[61:70] = 0
    result = evaluate_alarm(noisy, stream, primary)
    assert result['SeizurePredicted'] == 1
    assert abs(result['TimeToWarning_min'] - 25.0) < 1e-9

    # A discontinuity clears all prior evidence. With only 20 valid epochs
    # after the gap, no full 30-epoch firing-power window can be evaluated.
    broken = stream.copy()
    short_predictions = np.zeros_like(predictions)
    short_predictions[60:90] = 1
    broken.loc[70, 'Discontinuity'] = 1
    result = evaluate_alarm(short_predictions, broken, primary)
    assert result['SeizurePredicted'] == 0

    # Legacy rule remains reproducible only as a sensitivity analysis.
    result = evaluate_alarm(
        predictions, stream, ALARM_SENSITIVITY_SETTINGS['Legacy_90_consecutive'])
    assert result['SeizurePredicted'] == 1
    assert abs(result['TimeToWarning_min'] - 15.0) < 1e-9
    print('Causal alarm unit tests: PASS')


run_alarm_unit_tests()


## 5. Targeted configurations and safe output path


In [ ]:
TARGET_PAIRS = [
    ('RandomForest', 'F1_Handcrafted'),
    ('RandomForest', 'F7_All'),
    ('XGBoost', 'F1_Handcrafted'),
    ('XGBoost', 'F6_Handcrafted_wPLI'),
]
TARGET_FEATURES = sorted({feature for _, feature in TARGET_PAIRS})
TARGET_MODELS_BY_FEATURE = {
    feature: [model for model, selected_feature in TARGET_PAIRS if selected_feature == feature]
    for feature in TARGET_FEATURES
}
DISPLAY_FEATURE = {
    'F1_Handcrafted': 'F1 Handcrafted',
    'F6_Handcrafted_wPLI': 'F6 Handcrafted+pd-wPLI',
    'F7_All': 'F7 Handcrafted+PLI+pd-wPLI',
}
SAVE_STREAM_PREDICTIONS = True

if RESET_OUTPUT and OUTPUT_ROOT.exists():
    if OUTPUT_ROOT != Path('/kaggle/working/MBEC_V11B_TARGETED_STREAM'):
        raise RuntimeError(f'Refusing to reset unexpected path: {OUTPUT_ROOT}')
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / 'stream_predictions').mkdir(exist_ok=True)

print('Targeted configurations:')
for model, feature in TARGET_PAIRS:
    print(' ', model, DISPLAY_FEATURE[feature])
print('Output:', OUTPUT_ROOT)


## 6. Online-safe alarm generator and unit tests


In [ ]:
def run_online_safe_alarm(y_pred, stream, setting):
    """Generate alarms without using Phase, TargetPreictal, or EligibleForFAR.

    Only timestamp discontinuities and the fixed refractory rule affect alarm
    generation. Retrospective labels are attached to emitted alarms solely for
    subsequent scoring.
    """
    stream = stream.reset_index(drop=True)
    predictions = np.asarray(y_pred, dtype=np.int8)
    if len(predictions) != len(stream):
        raise ValueError('Prediction/stream length mismatch')
    if not np.isin(predictions, [0, 1]).all():
        raise ValueError('Alarm input predictions must be binary')

    mode, window_epochs, threshold, required_positive = _validate_alarm_setting(setting)
    history = deque(maxlen=window_epochs)
    refractory_until = None
    previous_start = None
    alarms = []

    for idx, (pred, row) in enumerate(zip(predictions, stream.itertuples(index=False))):
        start = row.EpochStart
        end = start + pd.Timedelta(seconds=EPOCH_SEC)
        gap = bool(row.Discontinuity)
        if previous_start is not None:
            elapsed = (start - previous_start).total_seconds()
            gap = gap or elapsed > EPOCH_SEC + 0.5 or elapsed <= 0
        previous_start = start

        if gap:
            history.clear()
            continue
        if refractory_until is not None and end < refractory_until:
            history.clear()
            continue

        if mode == 'consecutive':
            if pred == 0:
                history.clear()
            else:
                history.append(1)
        else:
            history.append(int(pred))

        positive_count = int(sum(history))
        if len(history) == window_epochs and positive_count >= required_positive:
            alarms.append({
                'Index': idx,
                'AlarmTime': end,
                'Phase': str(row.Phase).lower(),
                'EligibleForFAR': int(row.EligibleForFAR),
                'TargetPreictal': int(row.TargetPreictal),
                'WindowEpochs': window_epochs,
                'PositiveEpochs': positive_count,
                'FiringPower': positive_count / window_epochs,
            })
            history.clear()
            refractory_until = end + pd.Timedelta(seconds=REFRACTORY_SEC)
    return alarms


def evaluate_alarm_generator(y_pred, stream, setting, generation_mode):
    stream = stream.sort_values('Epoch', kind='stable').reset_index(drop=True)
    target = stream.loc[stream['TargetPreictal'].astype(bool)]
    if len(target) != 180:
        raise AssertionError(f'Expected 180 target-preictal epochs, got {len(target)}')
    seizure_onset = target['EpochStart'].max() + pd.Timedelta(seconds=EPOCH_SEC)
    valid_start = seizure_onset - pd.Timedelta(seconds=SPH_SEC + SOP_SEC)
    valid_end = seizure_onset - pd.Timedelta(seconds=SPH_SEC)

    if generation_mode == 'label_gated_v10':
        alarms = run_causal_alarm(y_pred, stream, setting)
    elif generation_mode == 'online_safe_gap_only':
        alarms = run_online_safe_alarm(y_pred, stream, setting)
    else:
        raise ValueError(generation_mode)

    true_alarms = [a for a in alarms if valid_start <= a['AlarmTime'] < valid_end]
    late_alarms = [a for a in alarms if valid_end <= a['AlarmTime'] < seizure_onset]
    false_alarms = [a for a in alarms if a['EligibleForFAR'] == 1]
    eligible_hours = float(stream['EligibleForFAR'].astype(bool).sum() * EPOCH_SEC / 3600.0)
    return {
        'AlarmGenerationMode': generation_mode,
        'SeizurePredicted': int(bool(true_alarms)),
        'FalseAlarms': len(false_alarms),
        'LateAlarms': len(late_alarms),
        'EligibleInterictalHours': eligible_hours,
        'FAR_per_h': len(false_alarms)/eligible_hours if eligible_hours > 0 else np.nan,
        'TimeToWarning_min': (
            (seizure_onset-true_alarms[0]['AlarmTime']).total_seconds()/60
            if true_alarms else np.nan),
        'TotalAlarms': len(alarms),
    }


def run_online_safe_alarm_unit_tests():
    onset = pd.Timestamp('2076-01-01T01:00:00Z')
    starts = pd.date_range(onset-pd.Timedelta(minutes=40), periods=240, freq='10s')
    stream = pd.DataFrame({
        'Epoch': np.arange(240), 'EpochStart': starts,
        'Phase': ['excluded']*30 + ['interictal']*30 + ['preictal']*180,
        'TargetPreictal': [0]*60 + [1]*180,
        'EligibleForFAR': [0]*30 + [1]*30 + [0]*180,
        'Discontinuity': 0,
    })
    pred = np.array([0]*60 + [1]*180)
    result = evaluate_alarm_generator(
        pred, stream, ALARM_SENSITIVITY_SETTINGS[PRIMARY_ALARM_NAME],
        'online_safe_gap_only')
    assert result['SeizurePredicted'] == 1
    broken = stream.copy()
    broken.loc[70, 'Discontinuity'] = 1
    short = np.zeros_like(pred); short[60:90] = 1
    result = evaluate_alarm_generator(
        short, broken, ALARM_SENSITIVITY_SETTINGS[PRIMARY_ALARM_NAME],
        'online_safe_gap_only')
    assert result['SeizurePredicted'] == 0
    print('Online-safe alarm unit tests: PASS')

run_online_safe_alarm_unit_tests()


## 7. Targeted execution, prediction export, and comparison


In [ ]:
def epoch_counts(y_true, y_pred):
    """Return TN, FP, FN and TP using a fixed binary label order."""
    tn, fp, fn, tp = confusion_matrix(
        np.asarray(y_true), np.asarray(y_pred), labels=[0, 1]
    ).ravel()
    return int(tn), int(fp), int(fn), int(tp)


# Smoke test prevents a missing/helper-order error before the expensive run.
assert epoch_counts([0, 0, 1, 1], [0, 1, 0, 1]) == (1, 1, 1, 1)


def save_stream_prediction_table(subject, model_name, feature_set, sid,
                                 stream, y_stream, owners, heldout_model):
    prediction_model_heldout = np.full(len(stream), int(heldout_model), dtype=np.int32)
    matched = owners >= 0
    prediction_model_heldout[matched] = owners[matched]
    violations = matched & (prediction_model_heldout != owners)
    if violations.any():
        raise AssertionError('Cross-fit owner/held-out-model mismatch')
    out = stream[[
        'Epoch', 'EpochStart', 'Phase', 'TargetPreictal',
        'EligibleForFAR', 'Discontinuity'
    ]].copy()
    out.insert(0, 'HeldOutSeizure', int(sid))
    out.insert(0, 'FeatureSet', feature_set)
    out.insert(0, 'Model', model_name)
    out.insert(0, 'Subject', subject)
    out['Prediction'] = np.asarray(y_stream, dtype=np.int8)
    out['EpochOwnerSeizure'] = owners.astype(np.int32)
    out['PredictionModelHeldOutSeizure'] = prediction_model_heldout
    out['PredictionSource'] = np.where(matched, 'crossfit_owner_model', 'target_fold_model')
    folder = OUTPUT_ROOT / 'stream_predictions' / subject
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f'{model_name}__{feature_set}__seizure_{int(sid):03d}.parquet'
    out.to_parquet(path, index=False)
    return path


def run_subject_v11(subject):
    train_pool, stream_pool = load_subject(subject)
    seizure_ids = sorted(train_pool.loc[train_pool['Class'] == 1, 'SeizureID'].unique())
    fingerprint_cols = feature_columns_for_blocks(train_pool, ['Handcrafted'])
    owner_map = build_interictal_owner_map(train_pool, fingerprint_cols)

    streams, owner_arrays, crossfit_qc = {}, {}, {}
    for sid in map(int, seizure_ids):
        stream = (stream_pool.loc[stream_pool['SeizureID'] == sid]
                  .sort_values('Epoch', kind='stable').reset_index(drop=True))
        hashes = epoch_fingerprints(stream, fingerprint_cols)
        eligible = stream['EligibleForFAR'].astype(bool).to_numpy()
        owners = np.full(len(stream), -1, dtype=np.int32)
        if eligible.any():
            owners[eligible] = np.array(
                [owner_map.get(int(h), -1) for h in hashes[eligible]], dtype=np.int32)
        matched = int((eligible & (owners >= 0)).sum())
        streams[sid] = stream
        owner_arrays[sid] = owners
        crossfit_qc[sid] = {
            'StreamEpochs': len(stream),
            'EligibleInterictalEpochs': int(eligible.sum()),
            'CrossFitMatchedInterictalEpochs': matched,
            'CrossFitUnmatchedInterictalEpochs': int(eligible.sum())-matched,
        }

    fold_rows, alarm_rows, selected_rows, tuning_rows, candidate_rows = [], [], [], [], []
    for feature_set in TARGET_FEATURES:
        blocks = FEATURE_SET_DEFINITIONS[feature_set]
        feature_cols = feature_columns_for_blocks(train_pool, blocks)
        models = TARGET_MODELS_BY_FEATURE[feature_set]
        caches = {model: {} for model in models}
        print(subject, feature_set, models, len(feature_cols), 'raw features', flush=True)

        for fold_index, sid in enumerate(map(int, seizure_ids)):
            train = train_pool.loc[train_pool['SeizureID'] != sid]
            test = (train_pool.loc[train_pool['SeizureID'] == sid]
                    .sort_values(['Class','Epoch'], kind='stable'))
            stream = streams[sid]
            fold_seed = RANDOM_SEED + fold_index
            prepared = prepare_fold(train, test, stream, feature_cols, blocks, fold_seed)
            for record in prepared['selected']:
                selected_rows.append({'Subject':subject, 'FeatureSet':feature_set,
                                      'HeldOutSeizure':sid, **record})
            for model_name in models:
                model, tuning, candidates = fit_classifier(
                    model_name, prepared['X_train'], prepared['y_train'],
                    prepared['groups_train'], fold_seed)
                y_test_pred = model.predict(prepared['X_test']).astype(np.int8)
                default_stream_pred = model.predict(prepared['X_stream']).astype(np.int8)
                caches[model_name][sid] = {
                    'fitted': (prepared['scaler'], prepared['selected_idx'], model),
                    'y_test': prepared['y_test'], 'y_test_pred': y_test_pred,
                    'default_stream_pred': default_stream_pred,
                }
                tuning_rows.append({'Subject':subject, 'FeatureSet':feature_set,
                                    'Model':model_name, 'HeldOutSeizure':sid, **tuning})
                for candidate in candidates:
                    candidate_rows.append({'Subject':subject, 'FeatureSet':feature_set,
                                           'Model':model_name, 'HeldOutSeizure':sid,
                                           **candidate})

        for model_name in models:
            cache = caches[model_name]
            for sid in map(int, seizure_ids):
                stream = streams[sid]
                owners = owner_arrays[sid]
                eligible = stream['EligibleForFAR'].astype(bool).to_numpy()
                result = cache[sid]
                y_stream = result['default_stream_pred'].copy()
                for owner in sorted(set(owners[eligible]) - {-1}):
                    idx = np.flatnonzero(eligible & (owners == owner))
                    y_stream[idx] = predict_pipeline(
                        stream.iloc[idx], feature_cols, cache[int(owner)]['fitted'])

                if SAVE_STREAM_PREDICTIONS:
                    save_stream_prediction_table(
                        subject, model_name, feature_set, sid,
                        stream, y_stream, owners, sid)

                tn, fp, fn, tp = epoch_counts(result['y_test'], result['y_test_pred'])
                for generation_mode in ['label_gated_v10', 'online_safe_gap_only']:
                    primary = evaluate_alarm_generator(
                        y_stream, stream,
                        ALARM_SENSITIVITY_SETTINGS[PRIMARY_ALARM_NAME],
                        generation_mode)
                    fold_rows.append({
                        'Subject':subject, 'Model':model_name, 'FeatureSet':feature_set,
                        'HeldOutSeizure':sid, 'AlarmRule':PRIMARY_ALARM_NAME,
                        'TN':tn, 'FP':fp, 'FN':fn, 'TP':tp,
                        **crossfit_qc[sid], **primary,
                    })
                    for alarm_name, setting in ALARM_SENSITIVITY_SETTINGS.items():
                        alarm = evaluate_alarm_generator(
                            y_stream, stream, setting, generation_mode)
                        alarm_rows.append({
                            'Subject':subject, 'Model':model_name,
                            'FeatureSet':feature_set, 'HeldOutSeizure':sid,
                            'AlarmRule':alarm_name,
                            'IsPrimary':int(alarm_name == PRIMARY_ALARM_NAME),
                            **crossfit_qc[sid], **alarm,
                        })
        del caches
        gc.collect()
    return fold_rows, alarm_rows, selected_rows, tuning_rows, candidate_rows


all_folds, all_alarms, all_selected, all_tuning, all_candidates = [], [], [], [], []
for subject in subjects:
    checkpoint = OUTPUT_ROOT / f'{subject}__COMPLETE.json'
    if checkpoint.exists() and not RESET_OUTPUT:
        print('Skipping completed subject:', subject)
        continue
    try:
        outputs = run_subject_v11(subject)
        all_folds.extend(outputs[0]); all_alarms.extend(outputs[1])
        all_selected.extend(outputs[2]); all_tuning.extend(outputs[3])
        all_candidates.extend(outputs[4])
        pd.DataFrame(outputs[0]).to_csv(OUTPUT_ROOT/f'{subject}__folds.csv', index=False)
        pd.DataFrame(outputs[1]).to_csv(OUTPUT_ROOT/f'{subject}__alarms.csv', index=False)
        pd.DataFrame(outputs[2]).to_csv(OUTPUT_ROOT/f'{subject}__selected.csv', index=False)
        pd.DataFrame(outputs[3]).to_csv(OUTPUT_ROOT/f'{subject}__tuning.csv', index=False)
        pd.DataFrame(outputs[4]).to_csv(OUTPUT_ROOT/f'{subject}__candidates.csv', index=False)
        checkpoint.write_text(json.dumps({'Subject':subject,'Status':'completed'}))
    except Exception:
        traceback.print_exc()
        raise

# Resume-safe aggregation from subject checkpoints.
def concat_subject_files(suffix):
    paths = sorted(OUTPUT_ROOT.glob(f'chb??__{suffix}.csv'))
    if not paths:
        raise FileNotFoundError(f'No subject files for suffix={suffix}')
    return pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)

folds = concat_subject_files('folds')
alarms = concat_subject_files('alarms')
selected = concat_subject_files('selected')
tuning = concat_subject_files('tuning')
folds.to_csv(OUTPUT_ROOT/'targeted_fold_results.csv', index=False)
alarms.to_csv(OUTPUT_ROOT/'targeted_alarm_sensitivity_folds.csv', index=False)
selected.to_csv(OUTPUT_ROOT/'targeted_selected_features.csv', index=False)
tuning.to_csv(OUTPUT_ROOT/'targeted_tuning_summary.csv', index=False)

subject_rows = []
primary = alarms.loc[alarms['IsPrimary'].astype(bool)]
for keys, group in primary.groupby(['Subject','Model','FeatureSet','AlarmGenerationMode']):
    subject, model, feature, mode = keys
    predicted = int(group['SeizurePredicted'].sum())
    seizures = len(group)
    false_alarms = int(group['FalseAlarms'].sum())
    hours = float(group['EligibleInterictalHours'].sum())
    subject_rows.append({
        'Subject':subject, 'Model':model, 'FeatureSet':feature,
        'AlarmGenerationMode':mode, 'Seizures':seizures,
        'SeizuresPredicted':predicted,
        'SeizureSensitivity':predicted/seizures,
        'FalseAlarms':false_alarms, 'EligibleInterictalHours':hours,
        'FAR_per_h':false_alarms/hours if hours else np.nan,
        'MeanTimeToWarning_min':group['TimeToWarning_min'].mean(),
    })
subject_summary = pd.DataFrame(subject_rows)
subject_summary.to_csv(OUTPUT_ROOT/'targeted_primary_subject_metrics.csv', index=False)

pooled_rows = []
for keys, group in primary.groupby(['Model','FeatureSet','AlarmGenerationMode']):
    model, feature, mode = keys
    predicted = int(group['SeizurePredicted'].sum()); seizures = len(group)
    false_alarms = int(group['FalseAlarms'].sum())
    hours = float(group['EligibleInterictalHours'].sum())
    patient_group = subject_summary.loc[
        subject_summary['Model'].eq(model) &
        subject_summary['FeatureSet'].eq(feature) &
        subject_summary['AlarmGenerationMode'].eq(mode)]
    pooled_rows.append({
        'Model':model, 'FeatureSet':feature, 'AlarmGenerationMode':mode,
        'Subjects':patient_group['Subject'].nunique(), 'Seizures':seizures,
        'SeizuresPredicted':predicted, 'SeizureSensitivity':predicted/seizures,
        'FalseAlarms':false_alarms, 'EligibleInterictalHours':hours,
        'FAR_per_h':false_alarms/hours if hours else np.nan,
        'MeanTimeToWarning_min':group['TimeToWarning_min'].mean(),
        'MacroMeanSeizureSensitivity':patient_group['SeizureSensitivity'].mean(),
        'MacroMeanFAR_per_h':patient_group['FAR_per_h'].mean(),
        'MacroMeanTTW_min':patient_group['MeanTimeToWarning_min'].mean(),
    })
pooled_summary = pd.DataFrame(pooled_rows)
pooled_summary.to_csv(OUTPUT_ROOT/'targeted_primary_pooled_macro.csv', index=False)
display(pooled_summary)

wide = pooled_summary.pivot_table(
    index=['Model','FeatureSet'], columns='AlarmGenerationMode',
    values=['SeizureSensitivity','FAR_per_h','MeanTimeToWarning_min'])
wide.to_csv(OUTPUT_ROOT/'label_gated_vs_online_safe_comparison.csv')
display(wide)

print('V11B targeted rerun complete:', OUTPUT_ROOT)
